# EDA - ACS PUMS (2024, 1-Year)

Same structure and four questions as `01_uci_adult_eda.ipynb`, so the two datasets are comparable in `03_comparison_eda.ipynb`.

Two expected differences from the UCI notebook:
- ACS PUMS is Census-imputed before release, so little is expected in the missing-values section.
- ACS uses numeric codes rather than text labels, so a decoding step is needed.

## 0. Setup

Helper functions are defined here so the same calculation is applied to both datasets.

In [1]:
import os
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, norm

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

RANDOM_SEED = 0
RESULTS_DIR = "../results/tables"
os.makedirs(RESULTS_DIR, exist_ok=True)


def cramers_v(confusion_matrix):
    """Cramér's V, used to measure how strongly two categorical columns are associated."""
    chi2 = chi2_contingency(confusion_matrix, correction=False)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    return (chi2 / (n * (min(r, k) - 1))) ** 0.5


def two_proportion_test(count1, n1, count2, n2):
    """Two-proportion z-test, used to compare two groups on a yes/no outcome."""
    p1, p2 = count1 / n1, count2 / n2
    p_pool = (count1 + count2) / (n1 + n2)
    se = (p_pool * (1 - p_pool) * (1 / n1 + 1 / n2)) ** 0.5
    z = (p1 - p2) / se
    return p1, p2, 2 * (1 - norm.cdf(abs(z)))


def pairwise_proportion_tests(df, group_col, outcome_col, positive_value):
    """Run a two-proportion z-test on every pair of groups. One row is returned per pair."""
    counts = pd.crosstab(df[group_col].astype(str), df[outcome_col].astype(str))
    counts["n"] = counts.sum(axis=1)

    rows = []
    for g1, g2 in combinations(counts.index, 2):
        p1, p2, p = two_proportion_test(
            count1=counts.loc[g1, positive_value], n1=counts.loc[g1, "n"],
            count2=counts.loc[g2, positive_value], n2=counts.loc[g2, "n"],
        )
        rows.append({
            "group_1": g1, "rate_1_pct": round(p1 * 100, 2), "n_1": counts.loc[g1, "n"],
            "group_2": g2, "rate_2_pct": round(p2 * 100, 2), "n_2": counts.loc[g2, "n"],
            "p_value": p, "verdict": "DIFFERENT" if p < 0.05 else "same",
        })
    return pd.DataFrame(rows).sort_values("p_value").reset_index(drop=True)


def proxy_screen(df, protected_col, feature_cols):
    """Measure how strongly each feature is associated with a protected attribute."""
    scores = {
        col: cramers_v(pd.crosstab(df[col], df[protected_col]))
        for col in feature_cols if col != protected_col
    }
    return pd.Series(scores).sort_values(ascending=False).round(3)


def group_rates(df, group_col, outcome_col, positive_value):
    """Percentage of each group receiving the positive outcome."""
    return pd.crosstab(df[group_col], df[outcome_col], normalize="index")[positive_value] * 100


def controlled_gap(df, control_col, group_col, outcome_col, positive_value,
                   advantaged, disadvantaged, min_group=0):
    """Compare two groups within levels of a control variable.

    The gap is measured separately inside each level of the control column and then
    averaged, weighted by how many records fall in each level. If the control variable
    explains the gap, the weighted gap should be much smaller than the raw gap.

    Levels holding fewer than `min_group` records of either group are excluded, because a
    near-empty level cannot support a comparison but can still distort the weighted average.

    Returns the per-level table and the weighted gap.
    """
    table = (
        df.groupby([control_col, group_col], observed=True)[outcome_col]
        .apply(lambda s: (s == positive_value).mean() * 100)
        .unstack(group_col)
    )
    counts = pd.crosstab(df[control_col], df[group_col]).reindex(table.index)

    table["gap"] = table[advantaged] - table[disadvantaged]
    table["n"] = counts.sum(axis=1)
    table["smallest_group"] = counts[[advantaged, disadvantaged]].min(axis=1)

    used = table[table["smallest_group"] >= min_group].dropna(subset=["gap"])
    weighted = (used["gap"] * used["n"]).sum() / used["n"].sum()

    return table, weighted

In [2]:
DATASET = "ACS PUMS (2024)"
POSITIVE = ">50K"

## 1. Data loading

Accessed via `folktables` using the standard `ACSIncome` task (Ding et al., 2021). Two handled issues: a full national pull exhausts memory, so a 10% subsample is drawn at retrieval with a fixed seed (reduces precision, not bias, as it ignores the protected attributes and target); and the Census renamed `RELP` to `RELSHIPP` from 2019, which folktables 0.0.12 predates, so an alias is set before the task is applied.

In [3]:
from folktables import ACSDataSource, ACSIncome

data_source = ACSDataSource(survey_year="2024", horizon="1-Year", survey="person")
acs_data = data_source.get_data(download=True, density=0.1, random_seed=RANDOM_SEED)

print("raw shape:", acs_data.shape)
print("SERIALNO sample:", acs_data["SERIALNO"].head(3).tolist())

raw shape: (343881, 286)
SERIALNO sample: ['2024GQ0001591', '2024GQ0001792', '2024GQ0002443']


In [4]:
if "RELP" not in acs_data.columns and "RELSHIPP" in acs_data.columns:
    acs_data["RELP"] = acs_data["RELSHIPP"]
    print("RELSHIPP aliased to RELP")
else:
    print("no alias needed; RELP present:", "RELP" in acs_data.columns)

RELSHIPP aliased to RELP


/tmp/ipykernel_77021/2043654623.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  acs_data["RELP"] = acs_data["RELSHIPP"]


`ACSIncome` applies its own filters (age > 16, hours > 0, minimum income), so the count drops sharply from the raw pull. The boolean label is converted to the same `>50K`/`<=50K` labels used in the UCI notebook.

In [5]:
features, label, group = ACSIncome.df_to_pandas(acs_data)

df_acs = features.copy()
df_acs["income"] = np.where(label.iloc[:, 0], ">50K", "<=50K")

# Recover raw dollar income. df_to_pandas resets the index, so alignment by label
# fails. Instead the ACSIncome row filter is re-applied to the raw frame and PINCP
# taken positionally, which matches df_to_pandas row-for-row and in the same order.
acs_filtered = ACSIncome._preprocess(acs_data)
assert len(acs_filtered) == len(df_acs), "row count mismatch after refiltering"
df_acs["PINCP"] = acs_filtered["PINCP"].to_numpy()

print("shape after ACSIncome filters:", df_acs.shape)
print("PINCP recovered:", df_acs["PINCP"].notna().all(),
      "| range:", df_acs["PINCP"].min(), "to", df_acs["PINCP"].max())
print()
print(df_acs["income"].value_counts())
print()
print((df_acs["income"].value_counts(normalize=True) * 100).round(2))

shape after ACSIncome filters: (175164, 12)
PINCP recovered: True | range: 110.0 to 1502000.0

income
<=50K    89202
>50K     85962
Name: count, dtype: int64

income
<=50K    50.92
>50K     49.08
Name: proportion, dtype: float64


## 2. Decoding

Codes present in the data are printed before mapping, because the aliased `RELP` column now holds `RELSHIPP` codes with a different numbering scheme.

In [6]:
for col in ["SEX", "RAC1P", "MAR", "COW"]:
    print(col, sorted(df_acs[col].dropna().unique()))

print()
print("RELP codes present:", sorted(df_acs["RELP"].dropna().unique()))
print("SCHL range:", df_acs["SCHL"].min(), "to", df_acs["SCHL"].max())
print("OCCP unique count:", df_acs["OCCP"].nunique())
print("POBP range:", df_acs["POBP"].min(), "to", df_acs["POBP"].max())

SEX [np.float64(1.0), np.float64(2.0)]
RAC1P [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0)]
MAR [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
COW [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0)]

RELP codes present: [np.float64(20.0), np.float64(21.0), np.float64(22.0), np.float64(23.0), np.float64(24.0), np.float64(25.0), np.float64(26.0), np.float64(27.0), np.float64(28.0), np.float64(29.0), np.float64(30.0), np.float64(31.0), np.float64(32.0), np.float64(33.0), np.float64(34.0), np.float64(35.0), np.float64(36.0), np.float64(37.0), np.float64(38.0)]
SCHL range: 1.0 to 24.0
OCCP unique count: 529
POBP range: 1.0 to 554.0


`SEX` and `RAC1P` are decoded below. The race mapping collapses the ACS categories onto the
five UCI categories, which is what makes the comparison between the two datasets possible.

| ACS `RAC1P` code | ACS meaning | Mapped to |
|---|---|---|
| 1 | White alone | White |
| 2 | Black or African American alone | Black |
| 3, 4, 5 | American Indian and/or Alaska Native | Amer-Indian-Eskimo |
| 6, 7 | Asian alone; Native Hawaiian and Other Pacific Islander alone | Asian-Pac-Islander |
| 8, 9 | Some other race alone; two or more races | Other |

The merging of codes 6 and 7 is needed because the UCI dataset records a single combined
`Asian-Pac-Islander` category. This is recorded as a limitation: the two datasets are made
comparable by reducing the finer 2024 scheme to the coarser 1994 one, which loses detail
that the ACS data actually contains.

In [7]:
sex_map = {1: "Male", 2: "Female"}

race_map = {
    1: "White",
    2: "Black",
    3: "Amer-Indian-Eskimo",
    4: "Amer-Indian-Eskimo",
    5: "Amer-Indian-Eskimo",
    6: "Asian-Pac-Islander",
    7: "Asian-Pac-Islander",
    8: "Other",
    9: "Other",
}

df_acs["sex"] = df_acs["SEX"].map(sex_map)
df_acs["race"] = df_acs["RAC1P"].map(race_map)

print("unmapped sex:", df_acs["sex"].isnull().sum())
print("unmapped race:", df_acs["race"].isnull().sum())
print()
print(df_acs["race"].value_counts())

unmapped sex: 0
unmapped race: 0

race
White                 116726
Other                  30199
Black                  13965
Asian-Pac-Islander     12260
Amer-Indian-Eskimo      2014
Name: count, dtype: int64


`SCHL` (education) and `OCCP` (occupation) hold high-cardinality numeric codes, banded below so Cramér's V stays readable an association across hundreds of codes would be inflated by category count alone.

In [8]:
def bucket_education(code):
    """Group the SCHL education codes into bands roughly comparable to the UCI levels."""
    if pd.isna(code):
        return "Unknown"
    code = int(code)
    if code <= 15:
        return "No high school diploma"
    if code <= 17:
        return "High school"
    if code <= 19:
        return "Some college"
    if code == 20:
        return "Associate"
    if code == 21:
        return "Bachelors"
    if code == 22:
        return "Masters"
    if code == 23:
        return "Professional"
    return "Doctorate"


# OCCP major groups, by the SOC code ranges published in the ACS data dictionary.
OCCP_BANDS = [
    (10, 440, "Management"), (500, 960, "Business/Finance"), (1005, 1240, "Computing/Maths"),
    (1300, 1560, "Engineering"), (1600, 1980, "Science"), (2001, 2060, "Community/Social"),
    (2100, 2180, "Legal"), (2205, 2555, "Education"), (2600, 2920, "Arts/Media"),
    (3000, 3550, "Healthcare practitioner"), (3601, 3655, "Healthcare support"),
    (3700, 3960, "Protective service"), (4000, 4160, "Food service"),
    (4200, 4255, "Cleaning/Maintenance"), (4330, 4655, "Personal care"),
    (4700, 4965, "Sales"), (5000, 5940, "Office/Admin"), (6005, 6130, "Farming/Fishing"),
    (6200, 6950, "Construction"), (7000, 7640, "Repair/Installation"),
    (7700, 8990, "Production"), (9005, 9760, "Transport/Material moving"),
]


def bucket_occupation(code):
    """Group the OCCP codes into major occupational bands."""
    if pd.isna(code):
        return "Unknown"
    code = int(code)
    for low, high, name in OCCP_BANDS:
        if low <= code <= high:
            return name
    return "Other/Unclassified"


df_acs["education"] = df_acs["SCHL"].apply(bucket_education)
df_acs["occupation"] = df_acs["OCCP"].apply(bucket_occupation)

print(df_acs["education"].value_counts())
print()
print(df_acs["occupation"].value_counts())

education
Bachelors                 41931
High school               41762
Some college              34668
Masters                   19423
Associate                 16038
No high school diploma    13354
Professional               4549
Doctorate                  3439
Name: count, dtype: int64

occupation
Management                   20519
Office/Admin                 18049
Sales                        15286
Transport/Material moving    12797
Education                    11991
Healthcare practitioner      11307
Business/Finance             10724
Food service                  8720
Production                    8678
Construction                  8056
Computing/Maths               6701
Cleaning/Maintenance          5745
Healthcare support            5270
Repair/Installation           5202
Personal care                 4751
Arts/Media                    4196
Engineering                   4046
Protective service            3670
Community/Social              3277
Science                       2

In [9]:
unclassified_pct = (df_acs["occupation"] == "Other/Unclassified").mean() * 100
print("unclassified occupation:", round(unclassified_pct, 2), "%")

if unclassified_pct > 5:
    print()
    print("band list looks incomplete. Codes not covered:")
    unmatched = df_acs.loc[df_acs["occupation"] == "Other/Unclassified", "OCCP"]
    print(sorted(unmatched.unique())[:50])

unclassified occupation: 0.37 %


Remaining columns are given readable names so the analysis code matches notebook 01. `relationship` is kept as its raw `RELSHIPP` code converted to text.

In [10]:
df_acs["marital-status"] = df_acs["MAR"].map({
    1: "Married", 2: "Widowed", 3: "Divorced", 4: "Separated", 5: "Never-married",
})
df_acs["workclass"] = df_acs["COW"].map({
    1: "Private-for-profit", 2: "Private-non-profit", 3: "Local-gov", 4: "State-gov",
    5: "Federal-gov", 6: "Self-emp-not-inc", 7: "Self-emp-inc", 8: "Without-pay",
    9: "Unemployed",
})
df_acs["hours-per-week"] = df_acs["WKHP"]
df_acs["age"] = df_acs["AGEP"]
df_acs["relationship"] = "code_" + df_acs["RELP"].astype("Int64").astype(str)

# POBP codes below 60 are US states and territories; higher codes are foreign birthplaces.
df_acs["birthplace-us"] = df_acs["POBP"] < 60

analysis_cols = [
    "age", "workclass", "education", "marital-status", "occupation", "relationship",
    "race", "sex", "hours-per-week", "birthplace-us", "income",
]
df_acs = df_acs[analysis_cols + ["PINCP", "RAC1P", "RELP", "OCCP", "SCHL", "POBP"]]

df_acs.head()

,age,workclass,education,marital-status,occupation,relationship,race,sex,hours-per-week,birthplace-us,income,PINCP,RAC1P,RELP,OCCP,SCHL,POBP
0,20.0,Private-for-profit,Some college,Never-married,Management,code_38,White,Male,30.0,True,<=50K,25000.0,1.0,38.0,310.0,19.0,1.0
1,19.0,Private-for-profit,Some college,Never-married,Production,code_38,White,Male,55.0,True,<=50K,19320.0,1.0,38.0,8030.0,18.0,39.0
2,38.0,Private-for-profit,No high school diploma,Married,Construction,code_37,White,Female,28.0,True,<=50K,11500.0,1.0,37.0,6230.0,15.0,1.0
3,27.0,Private-for-profit,Some college,Never-married,Transport/Material moving,code_37,White,Male,40.0,True,<=50K,10800.0,1.0,37.0,9600.0,19.0,1.0
4,21.0,Private-non-profit,Some college,Never-married,Personal care,code_38,White,Female,10.0,True,<=50K,9800.0,1.0,38.0,4640.0,19.0,12.0


### Findings : dataset shape

- The 10%-density subsample returns 343,881 raw records across 286 ACS columns. After the
  `ACSIncome` filters (age > 16, in the labour force, hours worked > 0, income > 100),
  175,164 records and 11 analysis columns remain.
- The target is close to balanced: 49.08% `>50K` against 50.92% `<=50K`. This is far closer
  to even than UCI's 76.1% / 23.9% split, as expected. The nominal $50,000 threshold now
  captures a much larger share of the working population than it did in 1994. Section 5.5
  tests directly how much of this shift is threshold-driven rather than a real change.

## 3. Missing values

The same checks are run as in notebook 01. Little is expected to be found, because ACS PUMS
is imputed before release. Any nulls that remain are expected to be structural rather than
non-response.

This section is kept despite the expected empty result, because the absence of non-response
missingness is itself the finding that separates the two datasets.

In [11]:
missing = df_acs[analysis_cols].isnull().sum()
missing = missing[missing > 0]

if len(missing) == 0:
    print("no missing values in the analysis columns")
else:
    print(missing)
    print()
    print((missing / len(df_acs) * 100).round(2))

no missing values in the analysis columns


In [12]:
# Where missingness is present, it is tested against the protected attributes as in notebook 01.
for col in list(missing.index):
    ct = pd.crosstab(df_acs["race"], df_acs[col].isnull())
    chi2, p, dof, expected = chi2_contingency(ct)
    print(f"{col} vs race: p={p:.6f}, V={cramers_v(ct):.4f}")

### Findings: missing values

- None of the 11 analysis columns contain missing values after the `ACSIncome` filters.
- This is a structural difference from UCI, not a data-quality improvement. ACS PUMS is
  imputed by the Census Bureau before public release, whereas UCI's missingness reflected
  genuine respondent non-response. This is why UCI required an explicit `"Unknown"`-category
  decision and ACS does not: no equivalent handling decision is needed for this dataset.

## 4. Proxy attributes

Cramér's V screen is applied as in notebook 01.

In [13]:
feature_cols = [
    "workclass", "education", "marital-status", "occupation", "relationship",
    "race", "sex", "birthplace-us", "income",
]

proxy_sex = proxy_screen(df_acs, "sex", feature_cols)
proxy_race = proxy_screen(df_acs, "race", feature_cols)

print("association with sex:")
print(proxy_sex)
print()
print("association with race:")
print(proxy_race)

association with sex:
occupation        0.451
workclass         0.161
income            0.138
education         0.108
marital-status    0.104
relationship      0.075
race              0.032
birthplace-us     0.012
dtype: float64

association with race:
birthplace-us     0.516
income            0.146
education         0.119
occupation        0.116
relationship      0.096
marital-status    0.082
workclass         0.048
sex               0.032
dtype: float64


In [14]:
# The strongest association for each protected attribute is inspected directly.
print("strongest proxy for sex:", proxy_sex.index[0])
print((pd.crosstab(df_acs[proxy_sex.index[0]], df_acs["sex"], normalize="index") * 100).round(2))
print()
print("strongest proxy for race:", proxy_race.index[0])
print((pd.crosstab(df_acs["race"], df_acs["birthplace-us"], normalize="index") * 100).round(2))

strongest proxy for sex: occupation
sex                        Female   Male
occupation                              
Arts/Media                  52.12  47.88
Business/Finance            54.13  45.87
Cleaning/Maintenance        39.37  60.63
Community/Social            65.09  34.91
Computing/Maths             26.92  73.08
Construction                 3.86  96.14
Education                   74.05  25.95
Engineering                 17.05  82.95
Farming/Fishing             24.67  75.33
Food service                56.40  43.60
Healthcare practitioner     74.91  25.09
Healthcare support          83.45  16.55
Legal                       54.94  45.06
Management                  42.61  57.39
Office/Admin                73.73  26.27
Other/Unclassified          13.78  86.22
Personal care               73.61  26.39
Production                  29.53  70.47
Protective service          24.31  75.69
Repair/Installation          4.34  95.66
Sales                       49.72  50.28
Science              

### Findings: proxy attributes

**Proxies for `sex`:**

| Feature | Cramér's V | Strength |
|---|---|---|
| occupation | 0.451 | moderate |
| workclass | 0.161 | weak |
| income (target) | 0.138 | weak |
| education | 0.108 | weak |
| marital-status | 0.104 | weak |
| relationship | 0.075 | negligible |
| race | 0.032 | negligible |
| birthplace-us | 0.012 | negligible |

- `occupation` is the strongest proxy in 2024, moderate rather than strong. No feature
  reaches UCI's `relationship` figure (V = 0.647).
- `relationship` has fallen from UCI's strongest proxy to negligible (0.075). A plausible
  reading is that the 2019+ RELSHIPP coding records household roles without the explicit
  sex-determined `Husband`/`Wife` split UCI used, so the variable no longer tracks sex. This
  is a plausible reading of the coding change, not a confirmed cause, and it changes which
  feature is removed alongside `sex` in the protected-attribute-removed experiment.

**Proxies for `race`:**

| Feature | Cramér's V | Strength |
|---|---|---|
| birthplace-us | 0.516 | moderate |
| income (target) | 0.146 | weak |
| education | 0.119 | weak |
| occupation | 0.116 | weak |
| relationship | 0.096 | negligible |
| marital-status | 0.082 | negligible |
| workclass | 0.048 | negligible |
| sex | 0.032 | negligible |

- `birthplace-us` is the strongest race proxy in both datasets, and is stronger in 2024
  (V = 0.516) than UCI's `native-country` was in 1994 (V = 0.415).

**What this means for the experiments:** in the protected-attribute-removed experiment,
`occupation` is the feature to remove alongside `sex` in 2024 (not `relationship`, as in
UCI), and `birthplace-us` alongside `race`, matching UCI's `native-country`.

## 5. Income gap by sex

The same method is used as in notebook 01. The raw gap is measured, and is then tested
against controls by comparing male and female rates inside groups of similar records.

In [15]:
rates_sex = group_rates(df_acs, "sex", "income", POSITIVE)
raw_gap_sex = rates_sex["Male"] - rates_sex["Female"]

print(rates_sex.round(2))
print()

ct_income_sex = pd.crosstab(df_acs["sex"], df_acs["income"])
chi2, p, dof, expected = chi2_contingency(ct_income_sex)
print("chi-square p-value:", p)
print("Cramér's V:", round(cramers_v(ct_income_sex), 4))
print()
print("raw gap (percentage points):", round(raw_gap_sex, 2))
print("ratio:", round(rates_sex["Male"] / rates_sex["Female"], 2))

sex
Female    41.91
Male      55.70
Name: >50K, dtype: float64

chi-square p-value: 0.0
Cramér's V: 0.1378

raw gap (percentage points): 13.79
ratio: 1.33


In [16]:
print(df_acs.groupby("sex", observed=True)["hours-per-week"].mean().round(2))

sex
Female    35.36
Male      39.81
Name: hours-per-week, dtype: float64


In [17]:
df_acs["hours_bucket"] = pd.cut(df_acs["hours-per-week"], bins=[0, 20, 35, 40, 45, 50, 100])

hours_table, gap_hours = controlled_gap(
    df_acs, control_col="hours_bucket", group_col="sex", outcome_col="income",
    positive_value=POSITIVE, advantaged="Male", disadvantaged="Female",
)

print(hours_table.round(2))
print()
print("raw gap:", round(raw_gap_sex, 2))
print("gap after controlling for hours:", round(gap_hours, 2))
print("% of raw gap remaining:", round(gap_hours / raw_gap_sex * 100, 1))

sex           Female   Male   gap      n  smallest_group
hours_bucket                                            
(0, 20]        11.71  20.50  8.79  24040            9533
(20, 35]       21.54  24.76  3.22  23965            9503
(35, 40]       50.98  58.64  7.66  88516           42195
(40, 45]       67.00  73.32  6.32  11136            4370
(45, 50]       71.97  76.09  4.11  14702            4874
(50, 100]      66.88  74.93  8.05  12805            3759

raw gap: 13.79
gap after controlling for hours: 6.85
% of raw gap remaining: 49.7


In [18]:
occ_table, gap_occ = controlled_gap(
    df_acs, control_col="occupation", group_col="sex", outcome_col="income",
    positive_value=POSITIVE, advantaged="Male", disadvantaged="Female",
)

print(occ_table.sort_values("gap", ascending=False).round(2))
print()
print("gap after controlling for occupation:", round(gap_occ, 2))
print("% of raw gap remaining:", round(gap_occ / raw_gap_sex * 100, 1))

sex                        Female   Male    gap      n  smallest_group
occupation                                                            
Production                  20.52  47.13  26.61   8678            2563
Sales                       28.87  52.29  23.42  15286            7600
Transport/Material moving   15.22  35.46  20.24  12797            2806
Repair/Installation         35.84  55.23  19.38   5202             226
Construction                26.69  45.87  19.19   8056             311
Protective service          41.82  60.66  18.84   3670             892
Farming/Fishing              7.94  25.18  17.24   1123             277
Cleaning/Maintenance         8.71  24.32  15.61   5745            2262
Education                   44.58  59.80  15.22  11991            3112
Legal                       72.66  87.77  15.11   2124             957
Engineering                 75.07  86.68  11.61   4046             690
Business/Finance            68.91  80.38  11.48  10724            4919
Health

The `relationship` control in notebook 01 needed two categories to be excluded, because
neither could support a comparison between men and women. The `min_group` argument is
therefore used here from the start, and the excluded categories are printed so that the
exclusion is visible rather than silent.

In [19]:
rel_table, gap_rel = controlled_gap(
    df_acs, control_col="relationship", group_col="sex", outcome_col="income",
    positive_value=POSITIVE, advantaged="Male", disadvantaged="Female", min_group=30,
)

print(rel_table.sort_values("gap", ascending=False).round(2))
print()
print("excluded:", rel_table[rel_table["smallest_group"] < 30].index.tolist())
print()
print("gap after controlling for relationship:", round(gap_rel, 2))
print("% of raw gap remaining:", round(gap_rel / raw_gap_sex * 100, 1))

sex           Female   Male    gap      n  smallest_group
relationship                                             
code_21        46.88  69.30  22.42  41265           20298
code_20        51.83  69.25  17.42  85486           41232
code_29        23.07  38.69  15.63   1135             398
code_32        24.59  37.65  13.06   1018             431
code_22        34.95  45.62  10.66   6201            2961
code_23        59.15  69.20  10.05    653             276
code_24        44.28  54.31  10.04    398             197
code_31        20.57  30.14   9.57    214              73
code_36        21.39  29.62   8.23   1967             907
code_27         6.40  13.19   6.78   1083             484
code_33        18.96  22.68   3.71   1690             733
code_34        27.09  30.60   3.51   3520            1543
code_30         7.58  10.61   3.04   1376             594
code_25        14.41  17.30   2.88  20214            8951
code_38         2.48   5.04   2.56   5379            2660
code_37       

### Findings: income gap by sex

**Raw gap:** the `>50K` rate is 55.70% for men and 41.91% for women, a gap of 13.79
percentage points, or a ratio of 1.33 to 1. The association is significant (chi-square
p < 0.001) with a Cramér's V of 0.138. The UCI figures were 19.45 points, ratio 2.78,
V = 0.215.

**Controlled comparisons:**

| Control variable | Gap remaining | % of raw gap | UCI comparison |
|---|---|---|---|
| hours worked | 6.85 points | 49.7% | UCI: 78.4% remaining |
| occupation | 13.72 points | 99.5% | UCI: 99.4% remaining |
| relationship (small groups excluded) | 15.11 points | 109.5% | UCI: 19.9% remaining |

- Hours worked explains noticeably more of the gap in 2024 than in 1994.
- Occupation explains almost none of it in either dataset, a consistent finding.
- `relationship` pushes the gap slightly past 100% in 2024, the opposite of UCI, where it
  explained about four fifths of the gap. This is consistent with the proxy-strength drop
  above: the 2024 household-role codes no longer track sex the way UCI's `Husband`/`Wife`
  categories did, so this control no longer isolates the same thing.

**Read against the threshold.** Any narrowing from 19.45 to 13.79 points must be read against
the threshold ceiling effect before it is called a real reduction in bias, because the
nominal $50,000 line places a much larger share of the 2024 population above it. Section 5.5
tests this directly by re-thresholding at an inflation-adjusted line.

## 5.5 Threshold sensitivity check

The UCI target used a nominal $50,000 income line in 1994. Applied unchanged to 2024 incomes,
that line sits far lower in real terms, so a much larger share of the population clears it and
group gaps compress mechanically. This is the ceiling effect flagged throughout: before the
narrowing in the sex gap (2.78 to 1.33) can be read as a real reduction in bias, it has to be
tested against a threshold that is comparable in real terms.

The 1994 $50,000 is inflation-adjusted to 2024 dollars using the BLS CPI-U annual averages,
and the sex gap is recomputed at the higher line. This check is only possible on the ACS data,
because the UCI dataset carries no dollar amounts and cannot be re-thresholded; the point is
recorded as an asymmetry between the two datasets.

In [20]:
# CPI-U annual averages (BLS, 1982-84=100): 1994 = 148.2, 2024 = 313.689.
CPI_1994, CPI_2024 = 148.2, 313.689
threshold_2024 = 50000 * CPI_2024 / CPI_1994
print(f"inflation-adjusted threshold: ${threshold_2024:,.0f}")

df_acs["income_adj"] = np.where(df_acs["PINCP"] > threshold_2024, ">50K", "<=50K")

rates_nominal = group_rates(df_acs, "sex", "income", POSITIVE)
rates_adj = group_rates(df_acs, "sex", "income_adj", POSITIVE)

sensitivity = pd.DataFrame({
    "threshold": ["nominal $50,000", f"adjusted ${threshold_2024:,.0f}"],
    "male_pct": [rates_nominal["Male"], rates_adj["Male"]],
    "female_pct": [rates_nominal["Female"], rates_adj["Female"]],
    "gap_pct_points": [rates_nominal["Male"] - rates_nominal["Female"],
                       rates_adj["Male"] - rates_adj["Female"]],
    "ratio": [rates_nominal["Male"] / rates_nominal["Female"],
              rates_adj["Male"] / rates_adj["Female"]],
})
sensitivity["dataset"] = DATASET
sensitivity.round(2)

inflation-adjusted threshold: $105,833


,threshold,male_pct,female_pct,gap_pct_points,ratio,dataset
0,"nominal $50,000",55.70,41.91,13.79,1.33,ACS PUMS (2024)
1,"adjusted $105,833",22.83,12.43,10.40,1.84,ACS PUMS (2024)


## 6. Income gap by race

The same pairwise testing is used as in notebook 01, so that the cluster structure found in
the 1994 data can be tested for in the 2024 data.

In [21]:
print(group_rates(df_acs, "race", "income", POSITIVE).sort_values(ascending=False).round(2))
print()
print(df_acs["race"].value_counts())

race
Asian-Pac-Islander    56.75
White                 53.04
Black                 38.19
Other                 36.62
Amer-Indian-Eskimo    34.66
Name: >50K, dtype: float64

race
White                 116726
Other                  30199
Black                  13965
Asian-Pac-Islander     12260
Amer-Indian-Eskimo      2014
Name: count, dtype: int64


In [22]:
race_tests_acs = pairwise_proportion_tests(df_acs, "race", "income", POSITIVE)
race_tests_acs

,group_1,rate_1_pct,n_1,group_2,rate_2_pct,n_2,p_value,verdict
0,Amer-Indian-Eskimo,34.66,2014,Asian-Pac-Islander,56.75,12260,0.000000e+00,DIFFERENT
1,Amer-Indian-Eskimo,34.66,2014,White,53.04,116726,0.000000e+00,DIFFERENT
2,Asian-Pac-Islander,56.75,12260,Other,36.62,30199,0.000000e+00,DIFFERENT
3,Asian-Pac-Islander,56.75,12260,Black,38.19,13965,0.000000e+00,DIFFERENT
4,Other,36.62,30199,White,53.04,116726,0.000000e+00,DIFFERENT
5,Black,38.19,13965,White,53.04,116726,0.000000e+00,DIFFERENT
6,Asian-Pac-Islander,56.75,12260,White,53.04,116726,4.662937e-15,DIFFERENT
7,Black,38.19,13965,Other,36.62,30199,1.517003e-03,DIFFERENT
8,Amer-Indian-Eskimo,34.66,2014,Black,38.19,13965,2.243209e-03,DIFFERENT
9,Amer-Indian-Eskimo,34.66,2014,Other,36.62,30199,7.643255e-02,same


### Findings: income gap by race

**Raw `>50K` rates:**

| Race | % earning >50K | n |
|---|---|---|
| Asian-Pac-Islander | 56.75% | 12,260 |
| White | 53.04% | 116,726 |
| Black | 38.19% | 13,965 |
| Other | 36.62% | 30,199 |
| Amer-Indian-Eskimo | 34.66% | 2,014 |

**Structure : the UCI two-cluster pattern does not replicate.** Every pairwise comparison is
significant (p < 0.001) except Amer-Indian-Eskimo against Other (p = 0.076, not significantly
different). White and Asian-Pac-Islander, statistically indistinguishable in UCI (p = 0.18),
are now significantly different from each other (p is approximately 5e-15), with
Asian-Pac-Islander overtaking White as the highest-earning group. The result reads as closer
to four distinct tiers than the clean two-cluster split found in 1994.

**Caveat on sample size:** the 2024 sample (175,164) is far larger than UCI's (48,842), so
smaller true differences reach significance more easily here. Some of the extra separation
may reflect statistical power rather than a larger underlying gap. This is recorded alongside
the result rather than reading every `DIFFERENT` verdict as a large effect.

**Effect on the race grouping.** The empirical basis for treating White and Asian-Pac-Islander as one
cluster, used in the UCI two-cluster framing, does not hold in 2024 data. The `race_binary`
(White against Non-White) grouping is retained for methodological consistency across the two
datasets, but the cluster-based grouping is no longer empirically supported in 2024 and is
recorded as a limitation rather than a neutral preprocessing choice.

## 7. Race binarisation

The same decision is applied as in notebook 01: `White` against all other categories
combined. Both groupings are computed so that the difference can be measured.

In [23]:
df_acs["race_binary"] = np.where(df_acs["race"] == "White", "White", "Non-White")
df_acs["race_cluster"] = np.where(
    df_acs["race"].isin(["White", "Asian-Pac-Islander"]), "Cluster A", "Cluster B"
)

binarisation_acs = {}
for col in ["race_binary", "race_cluster"]:
    rates = group_rates(df_acs, col, "income", POSITIVE)
    high, low = rates.idxmax(), rates.idxmin()
    binarisation_acs[col] = {
        "advantaged": high,
        "advantaged_rate_pct": round(rates[high], 2),
        "disadvantaged": low,
        "disadvantaged_rate_pct": round(rates[low], 2),
        "gap_pct_points": round(rates[high] - rates[low], 2),
        "ratio": round(rates[low] / rates[high], 3),
    }

pd.DataFrame(binarisation_acs).T

,advantaged,advantaged_rate_pct,disadvantaged,disadvantaged_rate_pct,gap_pct_points,ratio
race_binary,White,53.04,Non-White,41.15,11.89,0.776
race_cluster,Cluster A,53.39,Cluster B,37.01,16.39,0.693


## 8. Export

The same four tables are written as in notebook 01, with an `acs_` prefix and the same
column names, so that the comparison notebook can stack the two sets of tables directly.

In [24]:
proxy_table = pd.DataFrame({"cramers_v_sex": proxy_sex, "cramers_v_race": proxy_race})
proxy_table["dataset"] = DATASET
proxy_table.to_csv(f"{RESULTS_DIR}/acs_proxy_strength.csv")

controls_table = pd.DataFrame({
    "dataset": DATASET,
    "control": ["none (raw)", "hours worked", "occupation", "relationship (small groups excluded)"],
    "gap_pct_points": [raw_gap_sex, gap_hours, gap_occ, gap_rel],
})
controls_table["pct_of_raw_gap"] = controls_table["gap_pct_points"] / raw_gap_sex * 100
controls_table.round(2).to_csv(f"{RESULTS_DIR}/acs_sex_gap_controls.csv", index=False)

race_tests_out = race_tests_acs.copy()
race_tests_out["dataset"] = DATASET
race_tests_out.to_csv(f"{RESULTS_DIR}/acs_race_pairwise_tests.csv", index=False)

binarisation_out = pd.DataFrame(binarisation_acs).T
binarisation_out["dataset"] = DATASET
binarisation_out.to_csv(f"{RESULTS_DIR}/acs_race_binarisation.csv")

sensitivity.round(3).to_csv(f"{RESULTS_DIR}/acs_threshold_sensitivity.csv", index=False)

# The prepared frame is cached so that the comparison notebook does not need to pull the
# data again. The data/ directory is gitignored, so this copy stays local.
os.makedirs("../data/processed", exist_ok=True)
df_acs.to_csv("../data/processed/acs_2024_prepared.csv", index=False)

print(sorted(os.listdir(RESULTS_DIR)))

['acs_ci_baseline.csv', 'acs_intersectional.csv', 'acs_intersectional_matched.csv', 'acs_model_results.csv', 'acs_multiseed_results.csv', 'acs_multiseed_summary.csv', 'acs_proxy_strength.csv', 'acs_race_binarisation.csv', 'acs_race_pairwise_tests.csv', 'acs_sex_gap_controls.csv', 'acs_threshold_results.csv', 'acs_threshold_sensitivity.csv', 'intersectional_matched_comparison.csv', 'uci_ci_baseline.csv', 'uci_intersectional.csv', 'uci_model_results.csv', 'uci_proxy_strength.csv', 'uci_race_binarisation.csv', 'uci_race_pairwise_tests.csv', 'uci_sex_gap_controls.csv']


## 9. Summary

- **Shape:** 175,164 records and 11 analysis columns. The target is close to balanced at
  49.08% against 50.92%, in contrast to UCI's 76.1% / 23.9%. Section 5.5 shows how much of
  this shift is driven by the nominal threshold rather than a real change.
- **Missing values:** none after the `ACSIncome` filters. This reflects Census
  pre-imputation, a methodology difference from UCI's respondent non-response, not a
  data-quality difference, so no `"Unknown"`-category decision is needed here.
- **Proxy attributes:** `occupation` is the strongest sex proxy (V = 0.451), with
  `relationship` fallen to negligible; `birthplace-us` is the strongest race proxy (V = 0.516)
  and stronger than UCI's `native-country`. In the protected-attribute-removed experiment,
  `occupation` is removed alongside `sex` and `birthplace-us` alongside `race`.
- **Sex gap:** 13.79 points, ratio 1.33, down from UCI's 19.45 points and 2.78. This
  narrowing must be read against the threshold sensitivity result before being called a real
  reduction in bias.
- **Race gap:** the UCI two-cluster structure does not replicate. 2024 reads as four distinct
  tiers, with Asian-Pac-Islander now above White. `race_binary` is retained for cross-dataset
  consistency, but the cluster grouping is recorded as a limitation for 2024.
- **Carried forward to modelling:** `race_binary` as the protected race variable;
  `occupation` and `birthplace-us` as the proxy features to remove; and the threshold
  ceiling-effect caveat, which must accompany any sex-gap or race-gap narrowing claim.